# Missing-value imputation with TabFORGE

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SilenceX12138/TabFORGE/blob/master/docs/tutorial/notebooks/imputation/01_missing_value_imputation.ipynb)

> **Colab setup:** Open this notebook with the badge, then follow the [tutorial setup guide](https://silencex12138.github.io/TabFORGE/#/tutorials.md) to install `tabforge-ml[tutorials]` from PyPI in this runtime. Select a GPU for pretrained workloads.

## 1. Overview

This tutorial masks numerical and categorical cells in German Credit, reconstructs only those cells,
and verifies that observed values remain unchanged. Numerical trajectories are averaged as decoded
values. Categorical trajectory probabilities are averaged first, followed by one final argmax.

In [ ]:
%pip install "tabforge-ml[tutorials]"

## 2. Imports and setup

The default is a fast deterministic mock backend so the notebook executes offline. Set
`TABFORGE_TUTORIAL_MODEL_PATH=auto` to use the pinned pretrained Structure-aware Feature Encoder and backbone.

In [ ]:
import os
import tempfile
from pathlib import Path

from tabforge import (
    MODEL_ID,
    TabFORGEArchitectureConfig,
    TabFORGEEmbeddingConfig,
    TabFORGERuntimeConfig,
    TabFORGETrainingConfig,
)

RANDOM_STATE = 7
MAX_TRAIN_ROWS = int(os.environ.get("TABFORGE_TUTORIAL_MAX_ROWS", "256"))
MAX_STEPS = int(os.environ.get("TABFORGE_TUTORIAL_MAX_STEPS", "10"))
DEVICE = os.environ.get("TABFORGE_TUTORIAL_DEVICE", "auto")
MODEL_PATH = os.environ.get("TABFORGE_TUTORIAL_MODEL_PATH", "mock")
CHECKPOINT = None if MODEL_PATH == "mock" else "pretrained"

assert MODEL_ID == "tabforge-v1"

## 3. Prepare a real-world dataset

`TabularDataset` downloads German Credit from OpenML, keeps the mixed numerical/categorical schema,
and performs a reproducible stratified split. The tutorial caps the training rows to keep execution
short; increase `TABFORGE_TUTORIAL_MAX_ROWS` for a larger run.

In [ ]:
from tabcamel.data.dataset import TabularDataset

dataset = TabularDataset("credit-g", task_type="classification")
split = dataset.split("stratified", train_size=0.8, random_state=RANDOM_STATE)
train_set = split["train_set"]
test_set = split["test_set"]

X_train = train_set.X_df.iloc[:MAX_TRAIN_ROWS].reset_index(drop=True)
y_train = train_set.y_s.iloc[:MAX_TRAIN_ROWS].reset_index(drop=True)
X_test = test_set.X_df.reset_index(drop=True)
y_test = test_set.y_s.reset_index(drop=True)
categorical_features = tuple(dataset.categorical_feature_list)

print({"train": X_train.shape, "test": X_test.shape, "target": y_train.name})

## 4. Configure TabFORGE

`n_imputation_samples` controls independent conditional diffusion trajectories. More trajectories
stabilize numerical means and categorical mean probabilities.

In [ ]:
from tabforge import TabFORGEImputer

imputer = TabFORGEImputer(
    categorical_features=categorical_features,
    embedding_config=TabFORGEEmbeddingConfig(model_path=MODEL_PATH, n_folds=0 if MODEL_PATH == "mock" else 2),
    architecture_config=(
        TabFORGEArchitectureConfig(
            embedding_dimension=8 if MODEL_PATH == "mock" else None,
            decoder_layers=1, decoder_heads=2, decoder_ffn_factor=2,
            denoiser_layers=1, denoiser_heads=2, denoiser_ffn_factor=2,
        )
        if MODEL_PATH == "mock"
        else None
    ),
    training_config=TabFORGETrainingConfig(max_steps=MAX_STEPS, batch_size=128),
    runtime_config=TabFORGERuntimeConfig(device=DEVICE),
    n_imputation_samples=3,
    random_state=RANDOM_STATE,
)

## 5. Fit

The imputer learns the feature table without using the supervised target.

In [ ]:
imputer.fit(X_train, checkpoint=CHECKPOINT)

## 6. Impute

The public mask uses `True` for cells that should be regenerated. Omitting it automatically uses the
missing cells in `X`; supplying it also supports regenerating currently observed cells.

In [ ]:
complete = X_test.iloc[:16].copy()
incomplete = complete.copy()
incomplete.loc[incomplete.index[::2], "duration_months"] = float("nan")
incomplete.loc[incomplete.index[1::2], "checking_status"] = None
missing_mask = incomplete.isna()
print(f"Original missing values: {missing_mask.sum().sum()}")

automatic = imputer.transform(incomplete, random_state=RANDOM_STATE + 1)
imputed = imputer.transform(incomplete, mask=missing_mask, random_state=RANDOM_STATE + 1)
assert automatic.equals(imputed)

## 7. Inspect results

The requested cells are filled and every observed cell is restored exactly. Artificial masking lets us
measure numerical RMSE and categorical classification error against known values.

In [ ]:
import numpy as np

observed_equal = (
    imputed.where(~missing_mask).fillna("<missing>")
    == complete.where(~missing_mask).fillna("<missing>")
).to_numpy()[~missing_mask.to_numpy()].all()

numerical_rows = missing_mask["duration_months"]
categorical_rows = missing_mask["checking_status"]
numerical_rmse = np.sqrt(np.mean(
    (imputed.loc[numerical_rows, "duration_months"] - complete.loc[numerical_rows, "duration_months"]) ** 2
))
categorical_error = 1 - np.mean(
    imputed.loc[categorical_rows, "checking_status"].to_numpy()
    == complete.loc[categorical_rows, "checking_status"].to_numpy()
)
print({"numerical_rmse": numerical_rmse, "categorical_error": categorical_error, "observed_equal": bool(observed_equal)})
imputed.head()

## 8. Save, load, and sklearn interoperability

A restored imputer can process new tables with the same fitted schema.

In [ ]:
from sklearn.base import clone

assert clone(imputer).get_params()["n_imputation_samples"] == 3
with tempfile.TemporaryDirectory() as directory:
    checkpoint_path = imputer.save_checkpoint(Path(directory))
    restored = TabFORGEImputer.restore_from_checkpoint(checkpoint_path, device=DEVICE)
    restored_values = restored.transform(incomplete.iloc[:4], random_state=RANDOM_STATE + 2)

print({"checkpoint": checkpoint_path.name, "restored_rows": len(restored_values)})

## 9. Reproducibility notes

Keep the split, artificial missing mask, `random_state`, checkpoint revision, and trajectory count fixed
across methods. Report numerical RMSE and categorical error separately over repeated masks.